# Étape 5 — CNN-LSTM with Regime-as-Feature (walk-forward)
## MASI Hybrid Forecasting System

Compact CNN-LSTM (PyTorch, ~5k params), expanding walk-forward, regime ablation.

**HONEST RESULT — verdict `REJECTED` (RULE 8).** The CNN-LSTM has the best directional accuracy of any model (TEST DA 0.556) but does **not** beat ARIMA on Sharpe (1.06 vs 1.17), and the **regime feature does not help** (ablation −0.84 pp). Per RULE 8 the simple baselines (ARIMA / Random Forest) are recommended. See `etape5_walkforward_report.md`.

## 1. Run the walk-forward pipeline

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
from IPython.display import Image, display

import importlib.util as _ilu, sys as _sys; _spec = _ilu.spec_from_file_location('e5', '../scripts/05_cnn_lstm.py'); e5 = _ilu.module_from_spec(_spec); _sys.modules['e5'] = e5; _spec.loader.exec_module(e5)
e5.main()

## 2. Window-size scan (VAL)

In [ ]:
wf = json.load(open(os.path.join(e5.RESULTS_DIR, 'walkforward_metrics.json')))
print('selected L =', wf['selected_L'], '| architecture params =', wf['architecture_params'])
pd.DataFrame({L: {'VAL_DA': round(d['directional_accuracy'], 4), 'VAL_RMSE': round(d['rmse'], 6)}
              for L, d in wf['lscan'].items()}).T

## 3. TEST results — CNN-LSTM vs Étape 2 baselines

In [ ]:
rows = []
for nm, m in wf['baselines_test'].items():
    rows.append({'model': nm, 'DA': round(m['directional_accuracy'], 4),
                 'RMSE': round(m['rmse'], 6), 'Sharpe': round(m['sharpe_annualized'], 3),
                 'MDD': round(m['max_drawdown'], 3)})
for cfg, m in wf['cnn_test'].items():
    rows.append({'model': 'CNN-LSTM ' + cfg, 'DA': round(m['directional_accuracy'], 4),
                 'RMSE': round(m['rmse'], 6), 'Sharpe': round(m['sharpe_annualized'], 3),
                 'MDD': round(m['max_drawdown'], 3)})
display(pd.DataFrame(rows).set_index('model'))

## 4. Regime ablation — does the HMM regime help?

In [ ]:
a, b = wf['cnn_test']['base12'], wf['cnn_test']['base12+regime']
print(f"base12         : DA={a['directional_accuracy']:.4f}  Sharpe={a['sharpe_annualized']:+.3f}")
print(f"base12+regime  : DA={b['directional_accuracy']:.4f}  Sharpe={b['sharpe_annualized']:+.3f}")
print(f"\nablation gain (DA) = {wf['verdict']['ablation_gain_da']:+.4f}")
print('-> the regime feature does NOT help; it is DROPPED (RULE 6/8).')

## 5. Walk-forward fold stability

In [ ]:
rows = []
for f, d in wf['walk_forward']['base12'].items():
    m = d['metrics']
    rows.append({'fold': f, 'n': d['n'], 'DA': round(m['directional_accuracy'], 4),
                 'Sharpe': round(m['sharpe_annualized'], 3)})
print('base12 — DA stable across folds, Sharpe is NOT (one quarter loses money):')
display(pd.DataFrame(rows).set_index('fold'))

## 6. Verdict

In [ ]:
v = wf['verdict']
for k, val in v.items():
    print(f'  {k:22s}: {val}')
print('\n  OVERALL = REJECTED (RULE 8): the CNN-LSTM has the best DA but does not')
print('  beat ARIMA on Sharpe and is not stable on Sharpe -> ship ARIMA / Random Forest.')

## 7. Window-size scan plot

In [ ]:
Image(os.path.join(e5.PLOTS_DIR, 'etape5_lscan.png'))

## 8. CNN-LSTM vs baselines

In [ ]:
Image(os.path.join(e5.PLOTS_DIR, 'etape5_baseline_comparison.png'))

## 9. Walk-forward fold stability

In [ ]:
Image(os.path.join(e5.PLOTS_DIR, 'etape5_fold_stability.png'))

## 10. Cumulative TEST equity

In [ ]:
Image(os.path.join(e5.PLOTS_DIR, 'etape5_cumulative.png'))

## 11. Takeaways for Étape 6 (Backtesting)

| Insight | Implication |
|---------|-------------|
| CNN-LSTM verdict = REJECTED (RULE 8) | Ship the simple baselines, not the DL model |
| Best DA = CNN-LSTM (0.556); best Sharpe = ARIMA (1.17) | Backtest ARIMA + Random Forest as primary |
| Regime ablation = −0.84 pp | Drop the HMM regime feature |
| ARIMA Sharpe from 414 trades | Étape 6 must compute the Deflated Sharpe Ratio |
| DL did not beat simple models on frontier data | Honest negative result — the contribution is the rigorous methodology |